# Notebook 05 — Learning and resolution scaling

**Stability repair study.** The failed original run is preserved under `artifacts/notebook_05/learning-resolution-001/`; this notebook uses a fresh run ID and fresh paired models. Restart the **Python (ahr-go)** kernel and click **Run All**. The launch cell starts the actual experiment in detached tmux; subsequent cells monitor it and display the results. Closing the browser does not stop training.

This notebook asks three separate questions: does longer JEPA training improve semantic-local features beyond initialization; does higher image resolution help; and does a larger pretraining pool help? It compares semantic-local and ViT encoders with fresh paired initialization and a common EMA schedule.

**Planning runtime:** full staged profile **12–24 hours**, with a **24-hour hard limit**; learning-only profile **3–4 hours**, with a **4-hour hard limit**. These estimates use prior A100 measurements where available. These historical estimates predate the numerical repair; actual configuration calibration refines them before each stage. The 256×256 training configuration is not yet measured. The full profile may finish after the learning stage if its registered evidence gate does not pass. Increasing image/data size does not guarantee useful learning or faster computation.

**Inputs:** verified local Food101 archive and frozen Notebook04 provenance, current tagged notebook definitions, one A100 GPU. **Outputs:** stage plans, calibration and exact-resume certificates, training logs, immutable 300/1000-update snapshots, frozen-feature predictions, comparisons and native figures under `artifacts/notebook_05/<RUN_ID>/`. **Checkpoints:** `checkpoints/<RUN_ID>/stages/<stage>/`.

## 1. Study design

| Stage | Image size | Patch tokens | Pretraining images | Comparison |
|---|---:|---:|---:|---|
| Learning | 128×128 | 256 | 10,100 | 1,000-update schedule; initialization and snapshots at 300/1,000 |
| Resolution | 256×256 | 1,024 | Same 10,100 | Compare with learning-stage 128×128 controls |
| Data volume | 128×128 | 256 | Audited full pool: 65,600 | Compare with learning-stage 10,100-image controls |

Each stage uses **ViT and semantic-local × three paired seeds**, effective batch 128, FP32, and EMA starting 0.99 and rising to 1.0 over 1,000 updates. Patch size 8, width 192/depth 4 and the common JEPA predictor stay fixed. The larger-pool stage changes pretraining data only: every stage's classifier is fitted on the **same 10,100 labeled images**, evaluated on the **same 2,020 development images**, and uses the final twentieth probe epoch. Official test predictions remain reserved.

An intermediate 300-update snapshot belongs to the new 1,000-update learning-rate/EMA schedule. It is not interchangeable with Notebook04's final 300-update model. Resolution and data stages start from fresh initialization rather than continuing the learning-stage weights. Compare paired learning gains as well as final accuracies; an initialization advantage is not proof of better learning.

## 2. Registered learning gate

The full profile proceeds to resolution and data-volume stages only if the semantic encoder's final-minus-initial gain has a **positive lower bound on the across-seed 95% t interval**, with finite representations that are not exactly constant across development images. Low effective rank is advisory rather than an automatic failure. The three seeds remain the replication units.

If the gate does not pass, the notebook saves the completed learning results, marks later stages **NOT_RUN**, and finishes with **LEARNING_UNESTABLISHED**. This is a research result, not a software failure. Three seeds and reused development data provide exploratory evidence; neither a positive gate nor a negative result proves broad architectural superiority or impossibility.

## 3. Load only tagged definitions

Imports do not execute upstream training cells. Earlier notebook definitions and the failed original experiment evidence remain unchanged. The `semantic-stable-v1` arithmetic is installed in memory for every semantic encoder, including initialization controls, calibration, training, teachers and evaluation.

In [ ]:
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
from pathlib import Path
import nbformat, importlib.util, sys
ROOT = Path("/home/benjamin/workspace/ahr_go/fmm_jeppa").resolve()
_boot = {}
for _cell in nbformat.read(ROOT / "fmm_jepa/bootstrap.ipynb", as_version=4).cells:
    if _cell.cell_type == "code" and "ahr-export" in _cell.metadata.get("tags", []):
        exec(compile(_cell.source, "bootstrap#ahr-export", "exec"), _boot)
_boot["install_notebook_package"](ROOT)
_spec = importlib.util.spec_from_loader("fmm05_workflow", _boot["TaggedNotebookLoader"](ROOT / "experiments/05_scaling/workflow.ipynb"))
WORKFLOW = importlib.util.module_from_spec(_spec)
sys.modules[_spec.name] = WORKFLOW
_spec.loader.exec_module(WORKFLOW)


## 4. Choose the profile and inspect the estimate

Default `PROFILE="staged"` runs the registered sequence subject to the learning gate and original deadline. Set `PROFILE="learning_only"` before the first Run All for the 3–4 hour learning stage. Keep the same RUN_ID, profile, configuration and exported source to resume. Use a new RUN_ID for a genuinely different study; never alter an active run's clock or schedule.

The clock starts only in the launch cell below. This preview reads existing measurements; it performs no calibration or training.

In [ ]:
RUN_ID = "learning-resolution-stable-001"
PROFILE = "staged"  # "staged" or "learning_only"
CONFIG = WORKFLOW.study_configuration(RUN_ID, PROFILE)
PREVIEW = WORKFLOW.preview_study(ROOT, CONFIG)
WORKFLOW.display_preview(PREVIEW)


## 5. Launch or reconnect

**Run All starts training here.** The launcher first requires a passing repair certificate for the exact stable-attention definitions. The worker verifies local data/source identities, measures the actual configuration, checks that the fixed schedule fits the remaining deadline, and validates uninterrupted versus resumed full-state training before the long run. Calibration weights are discarded. It never silently shortens 1,000 updates or replaces the intended experiment with a smoke run.

The worker is `fmm-scaling-<RUN_ID>`. Attach with `tmux attach -t fmm-scaling-learning-resolution-stable-001`; detach with **Ctrl-b**, then **d**. An independent `fmm-scaling-deadline-<RUN_ID>` session enforces the original hard limit. The deadline includes calibration, training, evaluation and reporting and survives reconnection.

Checkpoints are saved every 50 updates, at epoch/stage boundaries and within 15 minutes. The latest two valid resumable states and the best recovery state are retained, together with the registered 300/1,000-update evaluation snapshots. Model, teacher, optimizer, scheduler, scaler state, RNGs, sampler position, configuration and metrics are restored together. Invalid state is an error, never permission to restart silently. The first numerical failure stops the run immediately; no failed attempt is skipped and no checkpoint containing skipped attempts can be resumed into this paired study.

In [ ]:
LAUNCH = WORKFLOW.launch_study(ROOT, CONFIG)
LAUNCH


## 6. Follow progress

This cell monitors the detached worker. If the notebook disconnects, reopen it and Run All with the same configuration: the launcher reconnects or resumes valid saved state. If the original clock is exhausted, the notebook reports incomplete evidence; it does not extend the run automatically. Current state, worker log and per-stage progress remain in the artifact directory. The monitor shows each arm’s saved updates and failed-attempt count, plus a calibration-based forecast for remaining training. Saved counts can lag the live update by up to 50 updates; the forecast excludes remaining evaluation and checkpoint I/O.

In [ ]:
COMPLETION = WORKFLOW.monitor_study(ROOT, RUN_ID)
COMPLETION


## 7. Results and interpretation

The result cells display only saved evidence from this run. Expect learning curves against updates and measured GPU time, per-seed initialization/training comparisons, resolution/data-volume effects, and time/memory tables. Missing stages remain explicitly NOT_RUN. Development-image count does not substitute for independent training seeds.

Storage is the persistent boot disk; instance-deletion policy is unverified. Before recycling or deleting the machine, copy final/best checkpoints off-instance and verify their hashes. No machine recycling, external publication or held-out evaluation is part of Run All.

In [ ]:
RESULT = WORKFLOW.display_results(ROOT, RUN_ID)
RESULT


## Repair validation and exact recovery

Long training requires a passing `artifacts/notebook_05/repair_validation/stability_certificate.json` for the current stable-attention exports. It records reproduction of the original failure, independent operator regressions, bounded tests from late failed-run checkpoints and fresh full-state exact resume. These diagnostic runs are separate from the fresh scientific study and their weights are not reused.

Each enabled stage also measures its actual runtime and validates four uninterrupted updates against reconstructed two-plus-two updates before starting long training. The user has authorized the repaired long run. Run All launches or reconnects to it using the same new run ID; it never restarts the failed original experiment. See `docs/notebook_05_usage.md` and `docs/notebook_05_protocol.md` for recovery instructions and evidence limits.
